# Lab 26.3: A Screening Score Is a Rank

Chapter 26, Casebook: Explainability Across Industries. Run every cell in order; no code needs to be written.

A synthetic hotline receives referrals about households with one to four children. A sparse logistic model (L1 penalty, the same family as the LASSO model in the Allegheny case) predicts a placement-like outcome for each child. You then turn its probabilities into a 1 to 20 score where each value holds 5 percent of child-referrals, show screeners the **maximum** score per referral, and screen in by default above 17 when a child is 16 or younger. **All data are synthetic.** Group A and group B differ only in how often their households appear in public-system records, a pattern the ACLU analysis of the Allegheny tool argued about; the numbers say nothing about any real place.

In [1]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(2026)
N = 12000                                     # referrals
rows = []
for r in range(N):
    group = rng.choice(["A", "B"], p=[0.6, 0.4])
    rec = 0.6 if group == "B" else 0.3        # B households appear more often in public records
    hh = dict(referral=r, group=group,
              prior_referrals=rng.poisson(1.0),
              jail_record=int(rng.random() < rec * 0.5),
              public_behavioral_health=int(rng.random() < rec),
              household_size=rng.integers(1, 5))
    need = rng.normal(0, 1)                   # unobserved true need, same distribution in both groups
    for c in range(hh["household_size"]):
        age = int(rng.integers(0, 18))
        z = -3.2 + 0.9 * need + 0.35 * hh["prior_referrals"] + 0.4 * (age < 5) + 0.25 * hh["jail_record"] \
            + 0.25 * hh["public_behavioral_health"] + rng.normal(0, 0.3)
        rows.append({**hh, "child_age": age, "under5": int(age < 5),
                     "outcome": int(rng.random() < 1 / (1 + np.exp(-z)))})
df = pd.DataFrame(rows)
print(df.shape, "outcome rate", round(df.outcome.mean(), 3))
print(df.groupby("group")[["jail_record", "public_behavioral_health", "outcome"]].mean().round(3))

(30051, 9) outcome rate 0.101
       jail_record  public_behavioral_health  outcome
group                                                
A            0.145                     0.302    0.096
B            0.307                     0.613    0.108


**Checkpoint 1.** Group B households carry public records about twice as often as group A. The outcome rate is somewhat higher in group B, because the records feed into it in this simulation; the unobserved need is the same in both groups.

In [2]:
FEATS = ["prior_referrals", "jail_record", "public_behavioral_health", "under5", "child_age", "household_size"]
train_ids, test_ids = train_test_split(df.referral.unique(), test_size=0.4, random_state=0)  # split by referral
tr, te = df[df.referral.isin(train_ids)], df[df.referral.isin(test_ids)].copy()
model = LogisticRegression(penalty="l1", solver="liblinear", C=0.5).fit(tr[FEATS], tr.outcome)
print(pd.Series(model.coef_[0], FEATS).round(3))
te["p"] = model.predict_proba(te[FEATS])[:, 1]
edges = np.quantile(te.p, np.linspace(0, 1, 21))
te["score"] = np.clip(np.searchsorted(edges, te.p, side="right"), 1, 20)

prior_referrals             0.332
jail_record                 0.250
public_behavioral_health    0.213
under5                      0.387
child_age                   0.006
household_size              0.020
dtype: float64


Note the split is by referral, so that children from the same household never sit on both sides. The Allegheny team found that a split that let siblings fall on both sides overstated the first version's accuracy (Chapter 3 calls this leakage).

## Table 1: what one step of score means

In [3]:
t1 = te.groupby("score").agg(p_min=("p", "min"), p_max=("p", "max"), outcome_rate=("outcome", "mean"), n=("p", "size"))
t1["band_width_points"] = (t1.p_max - t1.p_min) * 100
print(t1.round(3).to_string())

       p_min  p_max  outcome_rate    n  band_width_points
score                                                    
1      0.053  0.056         0.059  539              0.329
2      0.056  0.058         0.057  637              0.156
3      0.058  0.067         0.088  637              0.957
4      0.067  0.070         0.070  603              0.290
5      0.070  0.074         0.095  592              0.350
6      0.074  0.076         0.088  600              0.224
7      0.076  0.078         0.091  606              0.146
8      0.078  0.080         0.074  583              0.149
9      0.080  0.090         0.086  639              1.064
10     0.090  0.094         0.097  595              0.330
11     0.094  0.096         0.111  614              0.240
12     0.096  0.101         0.090  602              0.467
13     0.101  0.105         0.107  606              0.401
14     0.105  0.113         0.099  606              0.792
15     0.113  0.123         0.113  601              1.005
16     0.123  

**Checkpoint 2.** Low scores cover bands well under one percentage point wide; score 20 covers a band many times wider than any other. The score is a rank, not a probability.

## Table 2: maximum score per referral and the default screen-in

In [4]:
ref = te.groupby("referral").agg(group=("group", "first"), max_score=("score", "max"),
                                 any_young=("child_age", lambda a: (a <= 16).any()),
                                 n_children=("score", "size"))
ref["default_screen_in"] = (ref.max_score > 17) & ref.any_young
child = te.assign(default_child=(te.score > 17) & (te.child_age <= 16))
t2 = pd.DataFrame({
    "children flagged on their own score": child.groupby("group").default_child.mean(),
    "referrals flagged on the maximum score": ref.groupby("group").default_screen_in.mean(),
    "mean children per referral": ref.groupby("group").n_children.mean()})
print(t2.round(3).to_string())

       children flagged on their own score  referrals flagged on the maximum score  mean children per referral
group                                                                                                         
A                                    0.120                                   0.164                       2.534
B                                    0.189                                   0.242                       2.492


**Checkpoint 3.** Taking the maximum over children raises the share flagged for every group, and more for larger households. Compare the two groups in both columns: the gap comes from records that a household cannot change.

## Table 3: the reasons a screener never sees

For a linear model, each feature's contribution to the log-odds is its coefficient times its value. These are local reasons that could be shown next to the score.

In [5]:
case = te.sort_values("p").iloc[-1]
contrib = pd.Series(model.coef_[0] * case[FEATS].astype(float).values, FEATS).sort_values(ascending=False)
print("score", int(case.score), "predicted probability", round(case.p, 3))
print(contrib.round(3).to_string())

score 20 predicted probability 0.408
prior_referrals             1.658
under5                      0.387
jail_record                 0.250
public_behavioral_health    0.213
household_size              0.061
child_age                   0.000


## Hand in

Tables 1 to 3 and a draft of the screen a call screener should see for this referral: the score, its probability band, the top reasons, and one sentence on what the reasons do not show (for example, that the records describe contact with public systems, not need).